In [1]:
import json, time, re, random
import pandas as pd
from pathlib import Path

OUT = Path("../data_processed")
DOC = Path("../docs")
PROMPTS = Path("../prompts")

products = pd.read_csv(OUT / "product_facts.csv")
reviews = pd.read_csv(OUT / "reviews.csv")

# Day5: 合规规则
policy = json.loads((DOC / "policy_rules.json").read_text(encoding="utf-8"))

# Day6: 模板（这里先读取，后面如果你接LLM会用到）
tpl_title  = (PROMPTS / "title.txt").read_text(encoding="utf-8")
tpl_bullets= (PROMPTS / "bullets.txt").read_text(encoding="utf-8")
tpl_detail = (PROMPTS / "detail.txt").read_text(encoding="utf-8")
tpl_video  = (PROMPTS / "video.txt").read_text(encoding="utf-8")

# Day3: 关键词候选（你现在有 pos/neg candidates）
lex_pos = pd.read_csv(OUT / "keyword_lexicon_pos_candidates.csv")
lex_neg = pd.read_csv(OUT / "keyword_lexicon_neg_candidates.csv")

pos_map = (lex_pos.sort_values(["category","freq"], ascending=[True, False])
           .groupby("category")["keyword"].apply(lambda x: list(x.head(30))).to_dict())
neg_map = (lex_neg.sort_values(["category","freq"], ascending=[True, False])
           .groupby("category")["keyword"].apply(lambda x: list(x.head(30))).to_dict())

print("loaded:", products.shape, reviews.shape)

loaded: (8494, 13) (1093667, 4)


In [4]:
def safe(v):
    if pd.isna(v):
        return "未明确说明"
    return str(v)

def build_fact_sheet(row):
    lines = [
        f"product_id: {safe(row.get('product_id'))}",
        f"brand_name: {safe(row.get('brand_name'))}",
        f"product_name: {safe(row.get('product_name'))}",
        f"category: {safe(row.get('primary_category'))} / {safe(row.get('secondary_category'))} / {safe(row.get('tertiary_category'))}",
        f"price_usd: {safe(row.get('price_usd'))}",
        f"sale_price_usd: {safe(row.get('sale_price_usd'))}",
        f"rating_avg: {safe(row.get('rating_avg'))}",
        f"review_cnt: {safe(row.get('review_cnt'))}",
        f"ingredients: {safe(row.get('ingredients'))}",
        f"highlights: {safe(row.get('highlights'))}",
    ]
    return "\n".join(lines)

def compliance_check(text: str):
    t_low = text.lower()
    violations = []
    for rule in policy["rules"]:
        terms = rule.get("terms_zh", []) + rule.get("terms_en", [])
        hits = [term for term in terms if term.lower() in t_low]
        if hits:
            violations.append({
                "rule_id": rule["id"],
                "severity": rule["severity"],
                "matched_terms": hits[:20],
                "suggestion_zh": rule.get("suggestion_zh", "")
            })
    return violations

def keyword_coverage(text: str, keywords):
    """返回：命中了多少个关键词"""
    t = text.lower()
    hits = []
    for k in keywords:
        if str(k).lower() in t:
            hits.append(k)
    return hits

In [5]:
def sample_review_evidence(pid, n=8):
    sub = reviews.loc[reviews["product_id"] == pid, "review_text"].dropna()
    if len(sub) == 0:
        return []
    k = min(n, len(sub))
    return random.sample(list(sub), k)

def generate_one_variant(row, keywords, risk_terms, seed=0):
    """
    规则版生成：保证结构齐全，用于跑通多版本+评分链路
    """
    random.seed(seed)
    brand = safe(row.get("brand_name"))
    name  = safe(row.get("product_name"))
    cat   = safe(row.get("primary_category"))

    kw_pick = [k for k in keywords if isinstance(k, str)]
    random.shuffle(kw_pick)
    kw1 = kw_pick[0] if len(kw_pick)>0 else "highlights"
    kw2 = kw_pick[1] if len(kw_pick)>1 else "daily"

    # 1) title
    title = f"{brand} {name} | {kw1}"[:60]

    # 2) bullets (5条)
    bullets = [
        kw1[:40],
        kw2[:40],
        "absorbs quickly",
        "great for daily routine",
        "patch test if sensitive" if len(risk_terms)>0 else "easy to use"
    ]

    # 3) detail (结构段落)
    detail = (
        f"**痛点**：想要更适合{cat}的日常体验？\n"
        f"**解决**：围绕 {kw1} / {kw2} 的使用感描述。\n"
        f"**证据**：关键词参考：{kw1}, {kw2}；评分 {safe(row.get('rating_avg'))}；评论数 {safe(row.get('review_cnt'))}。\n"
        f"**适用**：更适合日常通勤与基础护理。\n"
        f"**注意**：{'敏感肌先局部测试，不适即停用' if len(risk_terms)>0 else '如有不适请停止使用'}"
    )

    # 4) video (3镜头)
    video = (
        f"Shot1\nVisual: problem scene\nVoiceover: Need an easier routine?\nOn-screen text: daily\n\n"
        f"Shot2\nVisual: texture + application\nVoiceover: {kw1}, {kw2}, absorbs quickly.\nOn-screen text: {kw1[:20]}\n\n"
        f"Shot3\nVisual: reviews + product\nVoiceover: Many users mention it. {'Patch test if sensitive.' if len(risk_terms)>0 else ''}\nOn-screen text: try"
    )

    return {"title": title, "bullets": bullets, "detail": detail, "video": video}

def generate_variants(product_id, n=5):
    row = products.loc[products["product_id"] == product_id]
    if row.empty:
        raise ValueError(f"product_id not found: {product_id}")
    row = row.iloc[0]

    category = safe(row.get("primary_category"))
    keywords = pos_map.get(category, [])
    risk_terms = neg_map.get(category, [])

    # 如果关键词为空，用 highlights 兜底（仍然是事实表信息）
    if not keywords:
        highlights = safe(row.get("highlights"))
        keywords = [x.strip().strip("'") for x in highlights.strip("[]").split(",") if x.strip()][:10]

    variants = []
    for i in range(n):
        v = generate_one_variant(row, keywords, risk_terms, seed=i)
        variants.append(v)

    ctx = {
        "product_id": product_id,
        "category": category,
        "fact_sheet": build_fact_sheet(row),
        "keywords": keywords[:10],
        "risk_terms": risk_terms[:10],
        "review_evidence": sample_review_evidence(product_id, n=8)
    }
    return ctx, variants

In [6]:
def readability_score(text: str):
    """
    简单可读性：长度惩罚 + 重复惩罚（MVP）
    """
    t = text.strip()
    length = len(t)
    # 长度太短/太长都扣分
    score = 100
    if length < 30:
        score -= 10
    if length > 600:
        score -= 10

    # 重复率（非常简化）：重复单词比例高就扣分
    words = re.findall(r"[a-zA-Z']+", t.lower())
    if len(words) >= 20:
        uniq = len(set(words))
        rep_ratio = 1 - (uniq / len(words))
        if rep_ratio > 0.4:
            score -= 15
    return max(0, score)

def score_variant(v, keywords):
    # 1) 合规：违规就扣分
    all_text = " ".join([v["title"], " ".join(v["bullets"]), v["detail"], v["video"]])
    violations = compliance_check(all_text)
    compliance_penalty = 15 * len(violations)  # 每条违规扣 15
    compliance_score = max(0, 100 - compliance_penalty)

    # 2) 关键词覆盖：命中几个关键词（越多越好，但上限）
    hits = keyword_coverage(all_text, keywords)
    coverage_score = min(100, len(set(hits)) * 15)  # 每命中一个+15，上限100

    # 3) 可读性（简单规则）
    read_score = readability_score(all_text)

    # 总分（可调权重）
    total = round(0.4*compliance_score + 0.35*coverage_score + 0.25*read_score, 2)

    return {
        "total": total,
        "compliance_score": compliance_score,
        "coverage_score": coverage_score,
        "readability_score": read_score,
        "violations": violations,
        "keyword_hits": list(sorted(set(hits)))[:20]
    }

In [7]:
# 选一个 product_id（用评论最多的更有意思）
top_pid = reviews.groupby("product_id").size().sort_values(ascending=False).head(1).index[0]
ctx, variants = generate_variants(top_pid, n=5)

scored = []
for i, v in enumerate(variants):
    s = score_variant(v, ctx["keywords"])
    scored.append({"variant_id": i, **s, **v})

scored_df = pd.DataFrame(scored).sort_values("total", ascending=False)
scored_df[["variant_id","total","compliance_score","coverage_score","readability_score","keyword_hits"]]

,variant_id,total,compliance_score,coverage_score,readability_score,keyword_hits
0,0,80.0,85,60,100,"[product, skin, use, ve]"
1,1,80.0,85,60,100,"[product, skin, use, ve]"
2,2,80.0,85,60,100,"[product, skin, use, ve]"
3,3,80.0,85,60,100,"[product, skin, use, ve]"
4,4,77.5,85,60,90,"[product, skin, use, ve]"


In [8]:
import json
from datetime import datetime

def append_jsonl(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

run = {
    "run_id": datetime.utcnow().isoformat(),
    "product_id": ctx["product_id"],
    "category": ctx["category"],
    "fact_sheet": ctx["fact_sheet"],
    "keywords": ctx["keywords"],
    "risk_terms": ctx["risk_terms"],
    "variants": [
        {
            "variant_id": i,
            "output": v,
            "score": score_variant(v, ctx["keywords"])
        }
        for i, v in enumerate(variants)
    ]
}

jsonl_path = OUT / "generated_samples.jsonl"
append_jsonl(jsonl_path, run)
print("appended to:", jsonl_path)

appended to: ../data_processed/generated_samples.jsonl


/var/folders/6_/_9mh8sxn79q7fm6bm97t_b5m0000gn/T/ipykernel_89008/2400713976.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_id": datetime.utcnow().isoformat(),


In [9]:
!ls -lh ../data_processed/generated_samples.jsonl | head

-rw-r--r--  1 wangfengyuan  staff   7.5K  2 26 19:00 ../data_processed/generated_samples.jsonl
